# 02 \u00b7 Train the LoRA adapter

Thin runner over `argubot.generate.training.train()`. It loads the config and examples, trains the
adapter (eval each epoch, gradient accumulation and warmup from the config), saves the **adapter
only** to `cfg['paths']['adapter_dir']`, and prints validation perplexity.

All logic lives in the tested package; the notebook only wires it together and reports numbers.

> **Runnable now:** if the real training data isn't built yet, the config cell falls back to the toy
> set so you can verify the whole path end to end. Those numbers are meaningless \u2014 rerun on real
> data (from `scripts/build_training_data.py`) for the actual adapter.

## Setup

Runs anywhere (Colab / SCIS cluster / local): finds the repo root, puts `src/` on the path, and
installs `torch`/`transformers`/`peft`/`accelerate` only if missing.

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

# Colab check that does NOT raise when the `google` package is absent (it does on the
# SCIS cluster / a local venv). find_spec on a dotted name raises if the parent is missing.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# On Colab, clone the repo once. Elsewhere (cluster / local) it is already present.
if IN_COLAB:
    REPO_URL = "https://github.com/diyadj/argubot.git"
    if not os.path.isdir("argubot"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("argubot")

# Work from the repo root so configs/config.yaml and data/... resolve, no matter which
# directory the kernel started in (e.g. notebooks/ on the cluster).
for d in [Path.cwd(), *Path.cwd().parents]:
    if (d / "pyproject.toml").exists():
        os.chdir(d)
        break

# Make `argubot` importable in THIS kernel by putting src/ on the path. A `pip install -e .`
# adds a .pth that a running kernel only reads at startup, so this avoids a restart.
src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)

# Install only what is missing. Regular installs are importable immediately; the cluster's
# PyTorch notebook ships Jupyter + CUDA but not torch. numpy/pyyaml are argubot's own deps.
pip_name = {"torch": "torch", "transformers": "transformers", "peft": "peft",
            "accelerate": "accelerate", "numpy": "numpy", "yaml": "pyyaml"}
need = [pip_name[m] for m in pip_name if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import argubot  # noqa: F401
print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB, "| argubot:", argubot.__file__)

## Config, seed, and data

Everything comes from `configs/config.yaml`. If the real `train_examples.jsonl` / `val_examples.jsonl`
aren't present, this falls back to the toy data and says so loudly.

In [ ]:
import os
import torch
from transformers import set_seed
from argubot.config import load_config

cfg = load_config("configs/config.yaml")
set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the real training data if it exists; otherwise fall back to the toy set so this notebook
# is runnable NOW as a wiring check, before Tricia/Diya's train_examples.jsonl lands.
train_path, val_path = cfg["paths"]["train_examples"], cfg["paths"]["val_examples"]
USING_TOY = not (os.path.exists(train_path) and os.path.exists(val_path))
if USING_TOY:
    toy = "data/toy/toy_train_examples.jsonl"
    cfg["paths"]["train_examples"] = toy
    cfg["paths"]["val_examples"] = toy
    print("!! Real training data not found -> using TOY data. WIRING CHECK ONLY;")
    print("!! the perplexity and adapter are meaningless. Build real data with")
    print("!! scripts/build_training_data.py, then re-run this notebook.")
else:
    print("Using real training data.")

t = cfg["training"]
eff = t["per_device_batch_size"] * t["grad_accum_steps"]
print(f"\ndevice            : {device}" + (f" / {torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
print(f"base_model        : {cfg['generator']['base_model']}")
print(f"epochs            : {t['num_epochs']}")
print(f"batch x grad_accum: {t['per_device_batch_size']} x {t['grad_accum_steps']} (effective {eff})")
print(f"learning_rate     : {t['learning_rate']}")
print(f"warmup_ratio      : {t['warmup_ratio']}")
print(f"max_seq_len       : {t['max_seq_len']}")
print(f"fp16              : {t['fp16']}")
print(f"max_train_examples: {t['max_train_examples']}")
print(f"adapter_dir       : {cfg['paths']['adapter_dir']}")

## Train

One call. Watch the `RECORD FOR REPORT ->` lines (trainable params, training time) and the per-epoch
eval loss. If the loss is `NaN`, set `training.fp16: false` in the config and rerun.

In [ ]:
from argubot.generate.training import train

# Loads the model, builds the LoRA adapter, trains with eval each epoch + grad accum + warmup
# from the config, and saves the adapter only. Prints trainable params and training time.
result = train(cfg)

## Validation perplexity

Uses the repo's token-weighted `perplexity` (reply tokens only), not the base notebook's version that
counted pad and prompt tokens.

In [ ]:
from torch.utils.data import DataLoader
from argubot.generate.training import ReplyOnlyCollator
from argubot.eval.metrics import perplexity

# Validation perplexity over reply tokens only, with every slot filled (the system-C conditions).
val_loader = DataLoader(
    result.val_dataset,
    batch_size=cfg["training"]["per_device_batch_size"],
    collate_fn=ReplyOnlyCollator(result.tokenizer.pad_token_id),
)
ppl = perplexity(result.model, val_loader)
print(f"RECORD FOR REPORT -> validation perplexity: {ppl:.2f}")
print("(measured with slots filled; say so in the report. On toy data this number is meaningless.)")

## Sample reply

Quick check that the trained adapter generates through the shared `build_prompt`. On toy data the
reply won't be good \u2014 this just confirms the adapter loads and generates.

In [ ]:
import torch
from argubot.schemas import Turn
from argubot.prompt import build_prompt

result.model.eval()
history = [Turn(speaker="user", text="School uniforms should be mandatory because they cut down on bullying.")]
prompt = build_prompt(
    history,
    claims=[],
    passages=["School uniforms reduce bullying because clothing differences matter less."],
    max_history_turns=cfg["prompt"]["max_history_turns"],
    max_passage_words=cfg["prompt"]["max_passage_words"],
)
enc = result.tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=cfg["training"]["max_seq_len"]).to(result.model.device)
with torch.no_grad():
    out = result.model.generate(
        **enc,
        do_sample=True,
        temperature=cfg["decoding"]["temperature"],
        top_p=cfg["decoding"]["top_p"],
        max_new_tokens=cfg["decoding"]["max_new_tokens"],
        pad_token_id=result.tokenizer.pad_token_id,
    )
reply = result.tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
print("REPLY:\n" + reply.strip())

## Summary to copy into the report log

In [ ]:
pct = 100 * result.trainable_params / result.total_params
print("=== 02_train summary ===")
print(f"data used        : {'TOY (wiring check)' if USING_TOY else 'real'}")
print(f"base model       : {result.model_name}")
print(f"device           : {result.device}")
print(f"trainable params : {result.trainable_params:,} / {result.total_params:,} = {pct:.3f}%")
print(f"train time       : {result.train_seconds:.1f}s")
print(f"val perplexity   : {ppl:.2f}")
print(f"adapter saved to : {result.adapter_dir}")
print("\nAlso record for the report: epochs, learning rate + the LR search you ran,")
print("batch x grad_accum, warmup, GPU, and any NaN / OOM issues and how you fixed them.")